

## Chapter 3 — Typed Scientific Tools (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

### Learning objectives
- Define tools with **Pydantic v2 contracts** (typed inputs, validated outputs).
- Enforce **deterministic outputs** and **timeout / error behavior**.
- Attach **provenance fields** (source, version, timestamp) to every tool result.
- Validate tool I/O so downstream chains never receive malformed science.

**Runtime / cost:** tool logic is local and free; the optional LLM agent demo uses your configured provider. ~5 min.

> **LangChain 1.x note:** This notebook uses the current tool API (`@tool` with Pydantic v2 `args_schema`, `langchain_core.tools`) and structured-output patterns. Tool definitions are deterministic and run locally; the LLM-calling demo is gated behind a config flag.

## Why typed tools matter for science

A hallucinated molecular weight or an unvalidated dose is worse than no answer. Scientific tools must:
1. **Reject bad input** (a malformed SMILES, a negative concentration).
2. **Return a predictable schema** so downstream steps can parse it.
3. **Record provenance** so a claim can be traced to a source + version.

Pydantic v2 gives us all three. This notebook builds a small, correct toolkit rather than a large, sloppy one.

### API Configuration

Tool logic is local. To run the optional agent demo at the end, set `RUN_LLM_DEMO=True` and provide a key for OPENAI / GROQ / GEMINI / ANTHROPIC, plus an optional HF token.

In [1]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

✅ API keys loaded for OPENAI (source: Colab Secrets)


In [2]:
import os

SEED = 42
MODEL_ID = os.getenv("LC4LSH_MODEL_ID", "gpt-5-nano")
RUN_LLM_DEMO = True  # False  # set True to invoke an agent over the tools
TOOL_TIMEOUT_S = 5.0  # hard timeout for any single tool call
RUN_METADATA = {
    "chapter": 3,
    "notebook": "typed_scientific_tools",
    "seed": SEED,
    "model": MODEL_ID,
}
print(RUN_METADATA)

{'chapter': 3, 'notebook': 'typed_scientific_tools', 'seed': 42, 'model': 'gpt-5-nano'}


## Package Installation and Setup

Pinned versions with upper bounds — Last validated: 2026-07-21 (see UPDATE_2026.md).

In [3]:
# @title Installing Python dependencies
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "pydantic>=2.9,<3" rdkit python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.6 MB/s eta 0:00:00


## 1. A shared provenance model

Every tool result carries provenance: which tool, which code version, when, and the source. This is the backbone of evidence-first tooling.

In [4]:
# @title Importing libraries
from datetime import datetime, timezone
from typing import Any, Optional
from pydantic import BaseModel, Field

TOOL_VERSION = "2026.07.21"


class Provenance(BaseModel):
    tool: str
    version: str = TOOL_VERSION
    timestamp_utc: str = Field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    source: str = "local"  # e.g., 'rdkit', 'pubchem', 'local'


class ToolResult(BaseModel):
    ok: bool
    data: Optional[Any] = None
    error: Optional[str] = None
    provenance: Provenance


print("Provenance + ToolResult schemas ready")

Provenance + ToolResult schemas ready


## 2. A validated chemistry tool

`molecular_properties` validates the SMILES (via RDKit), computes descriptors, and wraps the result in `ToolResult` with provenance. Bad input returns `ok=False` with a clear error instead of raising.

In [5]:
from pydantic import field_validator
from rdkit import Chem
from rdkit.Chem import Descriptors


class MolInput(BaseModel):
    smiles: str = Field(..., description="A valid SMILES string")

    @field_validator("smiles")
    @classmethod
    def _valid_smiles(cls, v):
        if not v or not v.strip():
            raise ValueError("empty SMILES")
        if Chem.MolFromSmiles(v) is None:
            raise ValueError(f"unparseable SMILES: {v!r}")
        return v


class MolProps(BaseModel):
    smiles: str
    mol_weight: float
    logp: float
    hbd: int
    hba: int


def compute_mol_props(inp: MolInput) -> ToolResult:
    try:
        mol = Chem.MolFromSmiles(inp.smiles)
        props = MolProps(
            smiles=inp.smiles,
            mol_weight=round(Descriptors.MolWt(mol), 3),
            logp=round(Descriptors.MolLogP(mol), 3),
            hbd=Descriptors.NumHDonors(mol),
            hba=Descriptors.NumHAcceptors(mol),
        )
        return ToolResult(
            ok=True,
            data=props.model_dump(),
            provenance=Provenance(tool="molecular_properties", source="rdkit"),
        )
    except Exception as e:
        return ToolResult(
            ok=False,
            error=str(e),
            provenance=Provenance(tool="molecular_properties", source="rdkit"),
        )


# quick checks
print(compute_mol_props(MolInput(smiles="CC(=O)Oc1ccccc1C(=O)O")).model_dump())

# Invalid SMILES: Pydantic validates at input — the error is raised BEFORE
# compute_mol_props() runs, so its try/except never sees it.
# Wrap in our own try/except to demonstrate gracefully.
try:
    compute_mol_props(MolInput(smiles="not_a_smiles"))
except Exception as e:
    print(f"ValidationError at input: {e}")

{'ok': True, 'data': {'smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'mol_weight': 180.159, 'logp': 1.31, 'hbd': 1, 'hba': 3}, 'error': None, 'provenance': {'tool': 'molecular_properties', 'version': '2026.07.21', 'timestamp_utc': '2026-07-28T12:51:23.480394+00:00', 'source': 'rdkit'}}
ValidationError at input: 1 validation error for MolInput
smiles
  Value error, unparseable SMILES: 'not_a_smiles' [type=value_error, input_value='not_a_smiles', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


[12:51:23] SMILES Parse Error: syntax error while parsing: not_a_smiles
[12:51:23] SMILES Parse Error: check for mistakes around position 3:
[12:51:23] not_a_smiles
[12:51:23] ~~^
[12:51:23] SMILES Parse Error: Failed parsing SMILES 'not_a_smiles' for input: 'not_a_smiles'


## 3. Timeout + error wrapper

A tool that hangs breaks an agent. Wrap every call with a hard timeout and normalize errors into `ToolResult`.

In [6]:
import concurrent.futures as cf


def run_with_timeout(fn, arg, timeout_s=TOOL_TIMEOUT_S) -> ToolResult:
    with cf.ThreadPoolExecutor(max_workers=1) as ex:
        fut = ex.submit(fn, arg)
        try:
            return fut.result(timeout=timeout_s)
        except cf.TimeoutError:
            return ToolResult(
                ok=False,
                error=f"timeout after {timeout_s}s",
                provenance=Provenance(tool=getattr(fn, "__name__", "tool")),
            )
        except Exception as e:
            return ToolResult(
                ok=False,
                error=f"{type(e).__name__}: {e}",
                provenance=Provenance(tool=getattr(fn, "__name__", "tool")),
            )


# simulate a slow tool
import time


def slow_tool(x):
    time.sleep(10)
    return ToolResult(ok=True, data=x, provenance=Provenance(tool="slow_tool"))


res = run_with_timeout(slow_tool, 1, timeout_s=1.0)
print(f"slow_tool ok={res.ok} error={res.error}")
res2 = run_with_timeout(compute_mol_props, MolInput(smiles="CCO"), timeout_s=5.0)
print(f"ethanol ok={res2.ok} mw={res2.data['mol_weight']}")

slow_tool ok=False error=timeout after 1.0s
ethanol ok=True mw=46.069


## 4. Expose as LangChain tools

Wrap the validated functions with `@tool` + `args_schema` so an agent gets structured, validated arguments.

In [7]:
from langchain_core.tools import tool


@tool(args_schema=MolInput)
def molecular_properties(smiles: str) -> dict:
    """Compute molecular weight, logP, H-bond donors/acceptors for a SMILES string. Returns a validated ToolResult with provenance."""
    res = run_with_timeout(compute_mol_props, MolInput(smiles=smiles))
    return res.model_dump()


print(molecular_properties.name, "->", molecular_properties.args)
out = molecular_properties.invoke({"smiles": "CC(=O)Oc1ccccc1C(=O)O"})
print(out["ok"], out["data"]["mol_weight"], out["provenance"]["tool"])

molecular_properties -> {'smiles': {'description': 'A valid SMILES string', 'title': 'Smiles', 'type': 'string'}}
True 180.159 molecular_properties


## 5. Optional: agent over the typed tools

Gated behind `RUN_LLM_DEMO`. Uses the modern `create_agent` API.

In [8]:
if RUN_LLM_DEMO:
    try:
        from langchain_openai import ChatOpenAI
        from langchain.agents import create_agent

        llm = ChatOpenAI(model=MODEL_ID, temperature=0)
        agent = create_agent(llm, tools=[molecular_properties])
        ans = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": "What is the molecular weight and logP of aspirin (CC(=O)Oc1ccccc1C(=O)O)?",
                    }
                ]
            }
        )
        print(ans["messages"][-1].content)
    except Exception as e:
        print(f"⚠️  agent demo failed: {type(e).__name__}: {str(e)[:120]}")
else:
    print("RUN_LLM_DEMO=False — set True to run the agent over typed tools.")

Molecular weight: 180.159 g/mol
logP (octanol/water): 1.31

Additional RDKit-based properties (optional): H-bond donors = 1, H-bond acceptors = 3.


## Limitations & safety
- ThreadPool timeout does **not** kill a stuck C-level call (e.g., a hung C extension); for hard isolation use a subprocess.
- RDKit descriptor values are deterministic for a given RDKit version; pin RDKit for reproducible numbers.
- Provenance records the tool + version but not the *model's* reasoning; pair with LangSmith tracing for full lineage.
- Never let a tool silently coerce a borderline SMILES/dose — fail loudly and abstain.

## Cleanup

---
### Further Reading

| Notebook | Relevance |
|----------|-----------|
| **Chapter 3 LangChain Components** | LangChain 1.x building blocks |
| **Chapter 5 Bounded Scientific Workflow** | Tools used under explicit step/cost budgets |
| **Chapter 5 Building Personal Assistants LangGraph and Agents** | Typed tools in LangGraph agents |


In [9]:
import gc

gc.collect()
print("🧹 done")

🧹 done


## Exercises

1. Why is returning `ok=False` with an error better than raising an exception inside an agent tool?
   <details><summary>Hint</summary>An exception aborts the agent loop; a structured error lets the agent recover, retry, or abstain gracefully.</details>
2. What two provenance fields would you add for a tool that queries PubChem?
   <details><summary>Hint</summary>A source identifier (CID/URL) and a retrieval timestamp, so the claim is traceable and cache-invalidatable.</details>
3. Why validate the SMILES in the `args_schema` validator *and* again inside the function?
   <details><summary>Hint</summary>Defense in depth: the agent path validates args, but a direct programmatic call could bypass the schema.</details>

### Task A — Add a dose-safety tool
Write `check_dose(drug: str, dose_mg: float, weight_kg: float)` with a Pydantic schema that rejects negative doses and returns a `ToolResult` flagging doses above a per-drug max (use a small lookup dict).

### Task B — Subprocess isolation
Rewrite `run_with_timeout` to use `multiprocessing.Process` so a hung C call is actually terminated. Compare behavior on `slow_tool`.

### Task C — Provenance round-trip
Modify `molecular_properties` to include the RDKit version (`rdkit.__version__`) in provenance. Assert it appears in the tool output.

### Task D — Schema drift test
Write a regression test that calls each tool with a known input and asserts the exact output keys. This catches silent schema changes when you refactor.